In [1]:
!nvidia-smi
import torch, os
print('torch    :', torch.__version__)
print('cuda     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu      :', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'vram     : {total:.1f} GB')
print('vcpu     :', os.cpu_count())

Sat Oct  3 09:50:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q cshogi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 112.7 MB/s eta 0:00:00a 0:00:01


In [3]:
#インストールできたかを厳しく確認する。ここを通らなければ先に進まない
import importlib, subprocess, sys

import numpy
print('python :', sys.version.split()[0])
print('numpy  :', numpy.__version__)

try:
    cshogi = importlib.import_module('cshogi')
except ImportError as exc:
    print('\n*** cshogi をインポートできません ***')
    print('理由:', exc)
    print()
    if 'numpy' in str(exc).lower():
        print('NumPy の ABI 不一致です。ランタイムを再起動してから')
        print('このノートブックをセル2から再実行してください。')
    else:
        print('上のインストール出力にビルドエラーが出ていないか確認してください。')
    print()
    print('pip の認識:')
    print(subprocess.run([sys.executable, '-m', 'pip', 'show', 'cshogi'],
                         capture_output=True, text=True).stdout or '  (未インストール)')
    raise SystemExit('cshogi が使えないため中断しました')

print('cshogi :', getattr(cshogi, '__version__', '(版数不明)'))
print('board  :', cshogi.Board().sfen())

# サブプロセスからも見えることを確認する。learn_loop は各段階を
# サブプロセスで実行するため、ここが通らないと学習ループで失敗する
check = subprocess.run([sys.executable, '-c', 'import cshogi; print(cshogi.Board().sfen())'],
                       capture_output=True, text=True)
assert check.returncode == 0, f'サブプロセスで失敗: {check.stderr}'
print('subprocess から見えることも確認:', check.stdout.strip())

python : 3.13.15
numpy  : 2.1.3
cshogi : (版数不明)
board  : lnsgkgsnl/1r5b1/ppppppppp/9/9/9/PPPPPPPPP/1B5R1/LNSGKGSNL b - 1
subprocess から見えることも確認: lnsgkgsnl/1r5b1/ppppppppp/9/9/9/PPPPPPPPP/1B5R1/LNSGKGSNL b - 1


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!git clone https://github.com/hirotakaster/shogi-ai-lab

Cloning into 'shogi-ai-lab'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 60 (delta 14), reused 60 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 204.20 KiB | 15.71 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [6]:
%cd "/content/shogi-ai-lab"

/content/shogi-ai-lab


In [7]:
!python -m shogi_ai.backends

cshogi backend OK (cshogi (version unknown))


In [8]:
!python -m unittest discover -s tests -v 2>&1 | tail -30

test_degenerate_logits_fall_back_to_uniform (test_mcts.SoftmaxTests.test_degenerate_logits_fall_back_to_uniform) ... ok
test_empty_legal_set (test_mcts.SoftmaxTests.test_empty_legal_set) ... ok
test_masks_to_legal_indices_and_sums_to_one (test_mcts.SoftmaxTests.test_masks_to_legal_indices_and_sums_to_one) ... ok
test_checkpoint_round_trip_restores_the_architecture (test_model_selection.CheckpointTests.test_checkpoint_round_trip_restores_the_architecture) ... ok
test_legacy_checkpoint_is_refused_with_a_clear_message (test_model_selection.CheckpointTests.test_legacy_checkpoint_is_refused_with_a_clear_message) ... ok
test_list_model_files_returns_only_pt_files (test_model_selection.CheckpointTests.test_list_model_files_returns_only_pt_files) ... ok
test_missing_directory_is_empty_not_an_error (test_model_selection.CheckpointTests.test_missing_directory_is_empty_not_an_error) ... ok
test_presets_round_trip_independently (test_model_selection.CheckpointTests.test_presets_round_trip_independ

In [9]:
import json
history = json.load(open('/content/drive/MyDrive/shogi-learn/history.json'))
for row in history:
    score = 'n/a' if row['score'] is None else f"{row['score']:.3f}"
    flag = 'promoted' if row['promoted'] else 'rejected'
    print(f"round {row['round']:3d}  score {score:>5s}  {flag:9s}  {row['seconds']:7.1f}s")

round   1  score 0.500  rejected     400.7s
round   2  score 0.650  promoted    1750.8s
round   3  score 0.475  rejected    1820.6s
round   4  score 0.625  promoted    2409.8s
round   5  score 0.550  promoted    2394.3s
round   6  score 0.575  promoted    2497.1s
round   7  score 0.537  rejected    2555.2s
round   8  score 0.613  promoted    7658.0s


In [10]:
!python -m shogi_ai.train_az \
    --data /content/drive/MyDrive/shogi-learn/selfplay_*.jsonl \
    --max-positions 2000000 \
    --epochs 8 --batch-size 1024 --lr 8e-3 \
    --preset large \
    --out /content/drive/MyDrive/shogi-learn/policy_value.pt

training on cuda
loading data
  /content/drive/MyDrive/shogi-learn/selfplay_001.jsonl: 47382 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_002.jsonl: 86620 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_003.jsonl: 126349 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_004.jsonl: 168808 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_005.jsonl: 208898 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_006.jsonl: 249804 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_007.jsonl: 290538 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_008.jsonl: 356582 positions so far
  /content/drive/MyDrive/shogi-learn/selfplay_009.jsonl: 422684 positions so far
422684 positions, 85 planes, 2187 policy outputs
new network: 15 blocks x 256ch, 18.59M params
mixed precision enabled
/content/shogi-ai-lab/shogi_ai/train_az.py:235: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()

In [ ]:
!python -m shogi_ai.learn_loop \
    --rounds 40 --games 500 \
    --simulations 800 --batch-size 64 \
    --processes 12 --backend cshogi --preset large \
    --epochs 8 --train-batch-size 1024 --lr 4e-3 \
    --max-positions 2000000 \
    --eval-games 40 --gate 0.55 \
    --data-dir /content/drive/MyDrive/shogi-learn \
    --out-model /content/drive/MyDrive/shogi-learn/policy_value.pt

starting from the existing checkpoint /content/drive/MyDrive/shogi-learn/policy_value.pt

plan
  data dir    : /content/drive/MyDrive/shogi-learn
  champion    : /content/drive/MyDrive/shogi-learn/policy_value.pt (found)
  reusing     : rounds 1(done), 2(done), 3(done), 4(done), 5(done), 6(done), 7(done), 8(done), 9(data)

round 1/40: already finished in an earlier run; skipping
round 2/40: already finished in an earlier run; skipping
round 3/40: already finished in an earlier run; skipping
round 4/40: already finished in an earlier run; skipping
round 5/40: already finished in an earlier run; skipping
round 6/40: already finished in an earlier run; skipping
round 7/40: already finished in an earlier run; skipping
round 8/40: already finished in an earlier run; skipping

round 9/40
reusing existing /content/drive/MyDrive/shogi-learn/selfplay_009.jsonl (66102 positions)

$ /usr/bin/python3 -m shogi_ai.train_az --data /content/drive/MyDrive/shogi-learn/selfplay_001.jsonl /content/drive/M